In [0]:
# Locations used by this notebook: the raw landing folder and the Bronze table.
from pyspark.sql import functions as F

RAW_PATH = "/Volumes/workspace/sa_market/raw/"
BRONZE_TABLE = "workspace.sa_market.bronze_rates"

In [0]:
# Reads every JSON file in the landing folder and adds the source file name and load time.
raw = (
    spark.read
    .option("multiLine", True)
    .json(RAW_PATH)
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingested_at", F.current_timestamp())
)
display(raw)

In [0]:
# Writes the raw rows to the Bronze Delta table.
raw.write.mode("overwrite").saveAsTable(BRONZE_TABLE)

In [0]:
%sql
-- Counts rows and date range per currency in the Bronze table.
SELECT quote, COUNT(*) AS row_count, MIN(date) AS first_day, MAX(date) AS last_day
FROM workspace.sa_market.bronze_rates
GROUP BY quote